# TradeFlow AI — nb5_xgboost

Trains the initial XGBoost rejection predictor.

In [ ]:
!pip install xgboost pandas numpy scikit-learn

In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

print("Generating mock dataset of 5,000 historical declarations...")
np.random.seed(42)
n_samples = 5000

crs_scores = np.random.normal(70, 15, n_samples)
crs_scores = np.clip(crs_scores, 0, 100)
correction_counts = np.random.poisson(1, n_samples)
agent_agreements = np.random.uniform(0.6, 1.0, n_samples)

# Probability of rejection goes up if CRS is low, corrections are high, or agreement is low
prob_reject = 1.0 / (1.0 + np.exp((crs_scores - 50) * 0.1 - correction_counts * 0.5 + (agent_agreements - 0.8) * 5))
labels = np.random.binomial(1, prob_reject)

df = pd.DataFrame({
    'crs_score': crs_scores,
    'correction_count': correction_counts,
    'agent_agreement': agent_agreements,
    'rejected': labels
})

X = df.drop('rejected', axis=1)
y = df['rejected']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 4,
    'eta': 0.1,
    'subsample': 0.8
}

print("Training XGBoost model...")
model = xgb.train(params, dtrain, num_boost_round=100)

preds = model.predict(dtest)
auc = roc_auc_score(y_test, preds)
acc = accuracy_score(y_test, (preds > 0.5).astype(int))

print(f"Test AUC: {auc:.4f}")
print(f"Test Accuracy: {acc:.4f}")

model.save_model("rejection_predictor.json")
print("Model saved to rejection_predictor.json")
